In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

def parse_result_text(text: str) -> dict:
    """Analisa o texto bruto de um resultado do ECoF e extrai informações detalhadas."""
    
    # Limpa o texto removendo quebras de linha extras e espaços
    text = " ".join(text.split())
    
    data = {
        'Original_Epithet': None,
        'Original_Genus': None,
        'Original_Author_Year': None,
        'Status': 'Unknown',
        'Accepted_Name': None,
        'Accepted_Author_Year': None,
        'Family': None,
        'Subfamily': None,
        'Type_Locality': None,
        'Type_Specimens': None,
        'Habitat': None,
        'Raw_Text': text
    }

    # --- 1. Extração do Nome Original, Gênero, Autor e Ano ---
    # Tenta capturar o início do registro: "epithet, Genus Author Year:Page"
    # Ex: "kleinii, Scomber Bloch [M. E.] 1793:86"
    match_original = re.match(r"^([^,]+),\s+([A-Z][a-z]+(?:\s*\(.+?\))?)\s+(.*?)\s+(\d{4}):\d+", text)
    
    if match_original:
        data['Original_Epithet'] = match_original.group(1).strip()
        data['Original_Genus'] = match_original.group(2).strip()
        # Captura autor e ano juntos para simplificar
        data['Original_Author_Year'] = f"{match_original.group(3).strip()} {match_original.group(4).strip()}"

    # --- 2. Extração do Status Atual e Nome Válido ---
    match_status = re.search(r"Current status:\s*(.*?)\.", text)
    if match_status:
        status_full = match_status.group(1).strip()
        
        if "Synonym of" in status_full:
            data['Status'] = 'Synonym'
            # Extrai o nome aceito e o autor do nome aceito
            match_accepted = re.search(r"Synonym of\s+([A-Z][a-z]+\s+[a-z]+)\s+(\(.*?\d{4}\))", status_full)
            if match_accepted:
                data['Accepted_Name'] = match_accepted.group(1).strip()
                data['Accepted_Author_Year'] = match_accepted.group(2).strip()
        
        elif "Valid as" in status_full:
            data['Status'] = 'Valid'
            match_valid = re.search(r"Valid as\s+([A-Z][a-z]+\s+[a-z]+)\s+(\(.*?\d{4}\))", status_full)
            if match_valid:
                data['Accepted_Name'] = match_valid.group(1).strip()
                data['Accepted_Author_Year'] = match_valid.group(2).strip()

        else:
            data['Status'] = 'Uncertain/Other'

    # --- 3. Extração da Família e Subfamília ---
    # Procura por padrões como "Carangidae: Caranginae."
    match_family = re.search(r"([A-Z][a-z]+idae)(?::\s*([A-Z][a-z]+inae))?\.", text)
    if match_family:
        data['Family'] = match_family.group(1)
        if match_family.group(2):
            data['Subfamily'] = match_family.group(2)

    # --- 4. Extração do Habitat ---
    match_habitat = re.search(r"Habitat:\s*(.*?)\.", text)
    if match_habitat:
        data['Habitat'] = match_habitat.group(1).strip()

    # --- 5. Extração da Localidade-Tipo (Type Locality) ---
    # A localidade-tipo geralmente está antes de "Syntypes:", "Holotype:", ou "Type catalog:" 
    # e depois da referência da publicação (ref. XXXX])
    # É complexo pois não tem um marcador fixo de início claro.
    
    # Remove o início (até a primeira referência) para facilitar a busca
    temp_text = re.sub(r"^.*?ref\. \d+\]", "", text)
    
    # Procura pelo texto entre o início limpo e os marcadores de tipo
    match_locality = re.search(r"\]?\s*(.*?)\.\s*(Syntypes:|Holotype:|Lectotype:|Neotype:|Type catalog:|Based on)", temp_text)
    if match_locality:
        locality = match_locality.group(1).strip()
        # Limpeza adicional para remover possíveis resíduos de referências
        if locality and not locality.startswith('•') and '[' not in locality:
             data['Type_Locality'] = locality

    # --- 6. Extração dos Espécimes-Tipo (Type Specimens) ---
    match_types = re.search(r"(Syntypes:|Holotype:|Lectotype:|Neotype:)\s*(.*?)\.", text)
    if match_types:
        data['Type_Specimens'] = match_types.group(2).strip()

    return data

def search_catalog_enhanced(query: str, search_type: str) -> pd.DataFrame:
    """
    Realiza buscas no Eschmeyer's Catalog of Fishes e extrai informações detalhadas.

    Args:
        query (str): O termo de busca (família, gênero ou espécie).
        search_type (str): "genus_family", "species_family", "species_genus", "species".

    Returns:
        pd.DataFrame: Um DataFrame detalhado com os resultados da busca.
    """
    base_url = "https://researcharchive.calacademy.org/research/ichthyology/catalog/fishcatget.asp"
    
    # Validação e construção dos parâmetros (mantido do script original)
    params = {'tbl': 'species'}
    if search_type == "genus_family":
        params['tbl'] = 'genus'
        params['family'] = query
    elif search_type == "species_family":
        params['family'] = query
    elif search_type == "species_genus":
        params['genus'] = query
    elif search_type == "species":
        parts = query.split(" ", 1)
        if len(parts) < 2:
            raise ValueError("Para o tipo 'species', o query deve conter gênero e espécie.")
        params['genus'] = parts[0]
        params['species'] = parts[1]
    else:
        raise ValueError("Tipo de busca inválido.")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(base_url, params=params, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Erro ao acessar a página: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, 'html.parser')
    results_p = soup.find_all('p', class_='result')

    if not results_p:
        print(f"Nenhum resultado encontrado para: '{query}'")
        return pd.DataFrame()

    # Processa cada resultado usando a função de parsing aprimorada
    all_data = []
    for p in results_p:
        text_content = p.get_text()
        parsed_data = parse_result_text(text_content)
        all_data.append(parsed_data)

    df = pd.DataFrame(all_data)
    
    # Reordena as colunas para uma visualização mais lógica
    columns_order = [
        'Status', 'Accepted_Name', 'Accepted_Author_Year', 
        'Original_Genus', 'Original_Epithet', 'Original_Author_Year', 
        'Family', 'Subfamily', 'Habitat', 'Type_Locality', 
        'Type_Specimens', 'Raw_Text'
    ]
    # Filtra para garantir que apenas colunas existentes sejam incluídas (caso a busca seja por gênero, alguns campos podem não existir)
    df = df[[col for col in columns_order if col in df.columns]]

    return df

# --- Script de Teste ---
if __name__ == '__main__':
    print("Executando script de teste aprimorado...\n")

    # Exemplo baseado na busca inicial do usuário sobre sinônimos de Alepes kleinii
    print("\n--- Buscando informações detalhadas sobre a espécie 'Alepes kleinii' e seus sinônimos ---")
    # A busca no ECoF por uma espécie válida retorna ela mesma E todos os seus sinônimos
    df_alepes = search_catalog_enhanced(query="Alepes kleinii", search_type="species")
    
    # Exibindo o resultado completo para análise
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 100)
    
    print(f"Total de registros encontrados (incluindo sinônimos): {len(df_alepes)}")
    print(df_alepes)

    # Exemplo adicional: Buscando espécies em um gênero específico
    print("\n--- Buscando espécies no gênero 'Caranx' ---")
    df_caranx = search_catalog_enhanced(query="Caranx", search_type="species_genus")
    print(f"\nTotal de espécies/sinônimos no gênero Caranx: {len(df_caranx)}")
    print(df_caranx[['Status', 'Original_Genus', 'Original_Epithet', 'Accepted_Name', 'Family']].head(10))

Executando script de teste aprimorado...


--- Buscando informações detalhadas sobre a espécie 'Alepes kleinii' e seus sinônimos ---
Total de registros encontrados (incluindo sinônimos): 12
     Status   Accepted_Name Accepted_Author_Year  Original_Genus  \
0   Synonym  Alepes kleinii         (Bloch 1793)          Caranx   
1   Synonym  Alepes kleinii         (Bloch 1793)          Caranx   
2   Synonym  Alepes kleinii         (Bloch 1793)          Caranx   
3   Synonym  Alepes kleinii         (Bloch 1793)          Caranx   
4   Synonym  Alepes kleinii         (Bloch 1793)           Selar   
5   Synonym  Alepes kleinii         (Bloch 1793)     Micropteryx   
6   Synonym  Alepes kleinii         (Bloch 1793)           Selar   
7   Synonym  Alepes kleinii         (Bloch 1793)          Caranx   
8   Synonym  Alepes kleinii         (Bloch 1793)          Caranx   
9     Valid  Alepes kleinii         (Bloch 1793)         Scomber   
10  Synonym  Alepes kleinii         (Bloch 1793)  Caranx (Atul

In [10]:
import requests
import pandas as pd
from IPython.display import display

def buscar_especie_salve(nome_especie: str):
    """
    Busca informações de uma espécie na API Salve do ICMBio e retorna um DataFrame.

    Args:
        nome_especie (str): O nome científico ou comum da espécie a ser buscada.

    Returns:
        pandas.DataFrame: Um DataFrame com os dados da espécie encontrada ou None se ocorrer um erro.
    """
    # URL base da API para a busca de espécies
    # O parâmetro 'q' é usado para passar o nome da espécie na consulta
    url_base = "https://salve.icmbio.gov.br/salve-api/api/v2/species/search"
    params = {'q': nome_especie}

    print(f"Buscando pela espécie: '{nome_especie}'...")

    try:
        # Realiza a requisição GET para a API
        response = requests.get(url_base, params=params)

        # Verifica se a requisição foi bem-sucedida (código de status 200)
        response.raise_for_status()

        # Converte a resposta JSON em um dicionário Python
        dados = response.json()

        # A API retorna uma lista de resultados. Vamos verificar se algo foi encontrado.
        if not dados:
            print(f"Nenhuma espécie encontrada com o nome '{nome_especie}'.")
            return None

        # Converte a lista de dicionários diretamente para um DataFrame do Pandas
        # A API já retorna os dados em um formato ideal para isso
        df = pd.DataFrame(dados)

        print(f"Sucesso! {len(df)} registro(s) encontrado(s).")
        return df

    except requests.exceptions.HTTPError as http_err:
        print(f"Erro HTTP ocorrido: {http_err}")
        print(f"Código de status: {response.status_code}")
        print(f"Resposta do servidor: {response.text}")
        return None
    except requests.exceptions.RequestException as req_err:
        print(f"Ocorreu um erro na requisição: {req_err}")
        return None
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")
        return None

# --- EXEMPLO DE USO ---

# 1. Buscando por um nome científico (Onça-pintada)
nome_cientifico = "Panthera onca"
df_onca = buscar_especie_salve(nome_cientifico)

# Exibe o DataFrame se a busca for bem-sucedida
if df_onca is not None:
    print("\n--- Informações para 'Panthera onca' ---")
    # Aumenta o número de colunas visíveis para não truncar a visualização
    pd.set_option('display.max_columns', None)
    display(df_onca)

print("\n" + "="*80 + "\n")

# 2. Buscando por um nome comum (Arara-azul)
nome_comum = "arara azul"
df_arara = buscar_especie_salve(nome_comum)

if df_arara is not None:
    print("\n--- Informações para 'arara azul' ---")
    # A API pode retornar mais de uma espécie para um nome comum
    display(df_arara)

print("\n" + "="*80 + "\n")

# 3. Exemplo de espécie não encontrada
nome_inexistente = "Dragão de komodo"
df_fail = buscar_especie_salve(nome_inexistente)

Buscando pela espécie: 'Panthera onca'...
Erro HTTP ocorrido: 403 Client Error: Forbidden for url: https://salve.icmbio.gov.br/salve-api/api/v2/species/search?q=Panthera+onca
Código de status: 403
Resposta do servidor: <!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,"Helvetica Neue",Arial,"Noto Sans",sans-serif,"Apple Color Emoji","Segoe UI Emoji","Segoe UI Symbol","Noto Color Emoji"}body{display:flex;flex-direction:column;height:100vh;min-height:100vh}.main-content{margin:8rem auto;padding-left:1.5rem;max-width:60rem}@media (width <= 720px){.main-

# Teste da API Eschmeyer integrada ao dataFishing

Agora vamos testar a nova API Eschmeyer integrada ao sistema dataFishing.

In [ ]:
import asyncio
import sys
import os

# Adicionar o caminho do dataFishing ao sys.path
datafishing_path = r"c:\Users\luanr\OneDrive - ASSOCIACAO INSTITUTO TECNOLOGICO VALE - ITV\github\dataFishing"
if datafishing_path not in sys.path:
    sys.path.append(datafishing_path)

from dataFishing.apis.eschmeyer import get_eschmeyer_data_batch

# Lista de espécies para teste
species_list = [
    "Alepes kleinii",
    "Caranx hippos", 
    "Scomber japonicus",
    "Thunnus albacares",
    "Rhinella marina"  # Não é peixe - deve retornar sem dados
]

print("Testando a API Eschmeyer integrada...")
print(f"Buscando dados para {len(species_list)} espécies:")
for species in species_list:
    print(f"  - {species}")
print()

# Executar busca
df_result = asyncio.run(
    get_eschmeyer_data_batch(
        species_list=species_list,
        verbose=True,
        time_delay=1,
        max_concurrent=3,
        output_folder="./test_output"
    )
)

print("\n" + "="*50)
print("RESULTADOS:")
print("="*50)
print(f"Total de registros retornados: {len(df_result)}")
print("\nColunas disponíveis:")
for col in df_result.columns:
    print(f"  - {col}")

print("\nDados encontrados:")
print(df_result[['Species_Name', 'Status', 'Accepted_Name', 'Family', 'Synonyms_Count']].to_string())

In [ ]:
# Teste usando CLI completo
import subprocess
import os

# Criar arquivo de espécies para teste
species_file = "test_species_eschmeyer.txt"
with open(species_file, 'w') as f:
    f.write("Alepes kleinii\n")
    f.write("Caranx hippos\n")
    f.write("Scomber japonicus\n")

print(f"Arquivo de espécies criado: {species_file}")

# Executar via CLI
print("\nExecutando via CLI...")
cmd = [
    "python", "-m", "dataFishing.cli",
    "--input", species_file,
    "--eschmeyer",
    "--output", "./test_eschmeyer_output",
    "--verbose"
]

try:
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=datafishing_path)
    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    print(f"Return code: {result.returncode}")
except Exception as e:
    print(f"Erro ao executar CLI: {e}")

# Limpar arquivo de teste
if os.path.exists(species_file):
    os.remove(species_file)
    print(f"\nArquivo de teste removido: {species_file}")

# Teste da lógica corrigida do NCBI

Agora vamos testar se o NCBI só procura sequências quando ambas as condições são atendidas.

In [ ]:
import asyncio
import sys
import os

# Adicionar o caminho do dataFishing ao sys.path
datafishing_path = r"c:\Users\luanr\OneDrive - ASSOCIACAO INSTITUTO TECNOLOGICO VALE - ITV\github\dataFishing"
if datafishing_path not in sys.path:
    sys.path.append(datafishing_path)

from dataFishing.apis.genbank import get_ncbi_data_batch

# Lista de espécies para teste
species_list = [
    "Alepes kleinii",
    "Caranx hippos"
]

print("="*60)
print("TESTE 1: Apenas taxonomia (sem download_sequences e sem gene_list)")
print("="*60)

df_result1 = asyncio.run(
    get_ncbi_data_batch(
        species_list=species_list,
        verbose=True,
        download_sequences=False,
        gene_list=[],
        output_folder="./test_output_1"
    )
)

print(f"\nColunas retornadas: {list(df_result1.columns)}")
print("\nDeveria ter apenas colunas de taxonomia, sem colunas de genes")
print(df_result1[['Species Name', 'TaxID', 'Family']].to_string())

In [ ]:
print("="*60)
print("TESTE 2: Com download_sequences=True mas sem gene_list")
print("="*60)

df_result2 = asyncio.run(
    get_ncbi_data_batch(
        species_list=species_list,
        verbose=True,
        download_sequences=True,
        gene_list=[],  # Lista vazia
        output_folder="./test_output_2"
    )
)

print(f"\nColunas retornadas: {list(df_result2.columns)}")
print("\nDeveria ter apenas colunas de taxonomia, sem buscar sequências")
print(df_result2[['Species Name', 'TaxID', 'Family']].to_string())

In [ ]:
print("="*60)
print("TESTE 3: Com gene_list mas download_sequences=False")
print("="*60)

df_result3 = asyncio.run(
    get_ncbi_data_batch(
        species_list=species_list,
        verbose=True,
        download_sequences=False,
        gene_list=['COI', '16S'],
        output_folder="./test_output_3"
    )
)

print(f"\nColunas retornadas: {list(df_result3.columns)}")
print("\nDeveria ter apenas colunas de taxonomia, sem buscar sequências")
print(df_result3[['Species Name', 'TaxID', 'Family']].to_string())

In [ ]:
print("="*60)
print("TESTE 4: Com download_sequences=True E gene_list fornecida")
print("="*60)

df_result4 = asyncio.run(
    get_ncbi_data_batch(
        species_list=species_list,
        verbose=True,
        download_sequences=True,
        gene_list=['COI', '16S'],
        output_folder="./test_output_4"
    )
)

print(f"\nColunas retornadas: {list(df_result4.columns)}")
print("\nDeveria ter colunas de taxonomia + colunas de genes + download")
print(df_result4.to_string())

In [ ]:
# Teste de correção do erro de importação
import sys
import os

# Adicionar o caminho do dataFishing ao sys.path
datafishing_path = r"c:\Users\luanr\OneDrive - ASSOCIACAO INSTITUTO TECNOLOGICO VALE - ITV\github\dataFishing"
if datafishing_path not in sys.path:
    sys.path.append(datafishing_path)

# Testar a importação do genbank após a correção
try:
    from dataFishing.apis.genbank import get_ncbi_data_batch, ncbi_taxonomy
    print("✅ Importação do genbank bem-sucedida!")
    
    # Testar outras importações também
    from dataFishing.apis.eschmeyer import get_eschmeyer_data_batch
    print("✅ Importação do eschmeyer bem-sucedida!")
    
    from dataFishing.utils import load_api_keys
    print("✅ Importação do utils bem-sucedida!")
    
    # Testar CLI
    from dataFishing.cli import main
    print("✅ Importação do CLI bem-sucedida!")
    
    print("\n🎉 Todas as importações funcionando corretamente!")
    
except Exception as e:
    print(f"❌ Erro na importação: {e}")
    print(f"Tipo do erro: {type(e).__name__}")

# Teste de correção dos erros de importação

Vamos verificar se as importações estão corretas agora.

In [ ]:
# Teste de importações das APIs
import sys
import os

# Adicionar o caminho do dataFishing ao sys.path
datafishing_path = r"c:\Users\luanr\OneDrive - ASSOCIACAO INSTITUTO TECNOLOGICO VALE - ITV\github\dataFishing"
if datafishing_path not in sys.path:
    sys.path.append(datafishing_path)

print("Testando importações das APIs...")

try:
    # Testar imports específicos
    import aiohttp
    print("✅ aiohttp importado com sucesso")
    
    from Bio import Entrez, SeqIO
    print("✅ Biopython importado com sucesso")
    
    from dataFishing.apis.genbank import get_ncbi_data_batch
    print("✅ NCBI genbank API importada com sucesso")
    
    from dataFishing.apis.bold import get_bold_data_batch
    print("✅ BOLD API importada com sucesso")
    
    from dataFishing.apis.eschmeyer import get_eschmeyer_data_batch
    print("✅ Eschmeyer API importada com sucesso")
    
    print("\n🎉 Todas as importações funcionando!")
    
except ImportError as e:
    print(f"❌ Erro de importação: {e}")
except Exception as e:
    print(f"❌ Erro geral: {e}")

In [ ]:
# Teste rápido para verificar se as APIs respondem
import asyncio

# Lista pequena para teste
test_species = ["Caranx hippos"]

print("Testando conexões das APIs...")

# Teste BOLD com tratamento melhorado de erro 503
print("\n--- Testando BOLD ---")
try:
    result_bold = asyncio.run(
        get_bold_data_batch(
            species_list=test_species,
            verbose=True,
            max_retries=2,  # Reduzir retries para teste
            retry_delay=3,  # Delay menor para teste
            time_delay=0.5,
            output_folder="./test_apis"
        )
    )
    print(f"BOLD: {len(result_bold)} registros retornados")
except Exception as e:
    print(f"BOLD: Erro - {e}")

# Teste NCBI
print("\n--- Testando NCBI ---")
try:
    result_ncbi = asyncio.run(
        get_ncbi_data_batch(
            species_list=test_species,
            verbose=True,
            max_retries=2,
            time_delay=0.5,
            output_folder="./test_apis",
            download_sequences=False,
            gene_list=[]
        )
    )
    print(f"NCBI: {len(result_ncbi)} registros retornados")
except Exception as e:
    print(f"NCBI: Erro - {e}")

print("\nTeste concluído!")